# Why do we need Runnables?

Imagine building a travel-planning assistant. A traveler gives us structured data such as the destination, number of days, and interests. First, a prompt template turns that dictionary into a useful planning request. Next, a model receives the request and drafts an itinerary.

These two steps do different jobs and currently expose different methods:

- `NakliPromptTemplate` uses `format(...)` and expects a dictionary.
- `NakliLLM` uses `predict(...)` and expects a finished prompt string.

That is manageable for two manual calls, but it becomes inconvenient when a workflow grows. Every new component can require a different method name, a different input shape, and extra glue code to connect the output of one step to the input of the next step. The developer must remember the correct order and manually pass values between components.

In [10]:
import random

In [11]:
class NakliLLM:
    def __init__(self, api_key=None):
        self.api_key = api_key

    def predict(self, prompt):
        responses = [
            f"I love {prompt}!",
            f"{prompt} is amazing!",
            f"I can't get enough of {prompt}!",
        ]
        return {f'response :  {random.choice(responses)}'}

In [12]:
model = NakliLLM(api_key="your_api_key_here")

In [13]:
result = model.predict("a 3-day itinerary for Kyoto")
print(result)

{"response :  I can't get enough of a 3-day itinerary for Kyoto!"}


In [14]:
class NakliPromptTemplate:
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def format(self, input_dict):
        return self.template.format(**input_dict)

In [15]:
template = NakliPromptTemplate(
    template="Create a {trip_length}-day itinerary for {destination}. The traveler enjoys {interests}. Include one practical tip for each day.",
    input_variables=["trip_length", "destination", "interests"]
)

In [16]:
prompt = template.format({"trip_length": 3, "destination": "Kyoto", "interests": "temples, local food, and quiet neighborhoods"})
print(prompt)

Create a 3-day itinerary for Kyoto. The traveler enjoys temples, local food, and quiet neighborhoods. Include one practical tip for each day.


In [17]:
result2 = model.predict(prompt)
print(result2)

{'response :  Create a 3-day itinerary for Kyoto. The traveler enjoys temples, local food, and quiet neighborhoods. Include one practical tip for each day. is amazing!'}


## The problem in this example

The two calls above work, but the workflow is written as separate instructions:

1. Call `template.format(...)` to create the prompt.
2. Call `model.predict(...)` with that prompt.
3. Add more special-case calls if we later introduce a parser, retriever, or another model.

The components are useful, but they do not share a common way to be executed. This makes the workflow harder to read, reuse, test, and extend. For example, a developer could accidentally pass the original trip dictionary directly to the model, or pass the model response to a template that expects `trip_length`, `destination`, and `interests`.

A **Runnable** solves this coordination problem by giving each step one common entry point, usually `invoke(...)`. The next notebook keeps the same scenario but makes both the prompt template and the model Runnables, so they can be connected into one predictable pipeline.

In [18]:
# We can see clearly that model invoking is different than prompt formatting. We can not use the same interface for both. This is where Runnables come into play. Runnables provide a unified interface for both prompt formatting and model invocation, allowing us to chain them together seamlessly.